[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/07_training_cost_interview/07_training_cost_interview.ipynb)

# 07 · 训练成本估算面试题（capstone）

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
把 6ND、MFU、Chinchilla 全部用到真实模型上：算 FLOPs、估时间与美元成本、判断算力分配，回答"多少卡训多久多少钱"。

**你将完成：**
1. 6ND 算真实 Pythia 的训练 FLOPs
2. 带 MFU 估训练时间（真实 A100/H100 规格）
3. Chinchilla 最优 token 数（看 GPT-3 训练不足）
4. 完整成本 drill：美元 + 显存可行性

> 数据：真实 Pythia config + 真实硬件/训练事实（Pythia 训练于 300B token 的 Pile）。

## 0 · config 管线 + 硬件规格

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS={"pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
        "pythia-6.9b":"https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
        "pythia-12b":"https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json"}
def load_config(m):
    p=os.path.join(CACHE,f"{m}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(MODELS[m],p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
def param_count(c):
    L,h,V,I=c["L"],c["h"],c["V"],c["I"]; return 2*V*h+L*(4*h*h+4*h+2*h*I+(I+h)+4*h)+h
# 真实硬件峰值算力 (bf16/fp16, TFLOP/s)
HW = {"A100": 312e12, "H100": 990e12, "RTX4090": 165e12}
PRICE = {"A100": 1.5, "H100": 3.0, "RTX4090": 0.4}   # 美元/卡时（粗略市价）
print("硬件:", {k:f'{v/1e12:.0f}T' for k,v in HW.items()})

## 1 · 6ND：真实 Pythia 训练 FLOPs

Pythia 全系列都训练于 Pile 的 300B token。用 6ND 算各模型训练计算量。

In [ ]:
D = 300e9   # Pythia 训练 token 数（真实事实）
def train_flops(N, D): return 6*N*D
for m in ["pythia-1.4b","pythia-6.9b","pythia-12b"]:
    N=param_count(load_config(m))
    C=train_flops(N, D)
    print(f"{m:12s} N={N/1e9:5.1f}B  训练 FLOPs = 6ND = {C:.2e}")
print("\n=> 12b 训练约 2e22 FLOPs，这就是为什么要成百上千张卡训几个月")

## 2 · 带 MFU 估训练时间

时间 = 6ND / (峰值 · MFU · 卡数)。MFU 取真实的 ~40%。

In [ ]:
def train_days(N, D, gpu, n_gpu, mfu=0.4):
    C=6*N*D; eff=HW[gpu]*mfu*n_gpu
    return C/eff/86400
N=param_count(load_config("pythia-12b"))
for gpu,n in [("A100",256),("H100",256),("A100",1024)]:
    d=train_days(N, D, gpu, n)
    print(f"pythia-12b 在 {n}×{gpu} (MFU 40%): {d:.1f} 天")
print("\n=> 卡越多越快；H100 比 A100 快 ~3x（峰值高）")
# 用峰值(MFU=100%)会低估
print(f"若按峰值(MFU=100%)算: {train_days(N,D,'A100',256,1.0):.1f} 天 -> 严重低估真实时间")

## 3 · Chinchilla：GPT-3 训练不足了吗

Chinchilla 最优 D≈20N。看 GPT-3(175B, 300B token) 偏离多少。

In [ ]:
def chinchilla_tokens(N): return 20*N
gpt3_N=175e9; gpt3_D=300e9
opt_D=chinchilla_tokens(gpt3_N)
print(f"GPT-3: N=175B, 实际 D=300B token")
print(f"Chinchilla 最优 D = 20N = {opt_D/1e12:.1f}T token")
print(f"GPT-3 实际只用了最优的 {gpt3_D/opt_D:.1%} -> 严重训练不足(undertrained)")
print(f"\nChinchilla(70B, 1.4T token): D/N = {1.4e12/70e9:.0f} ≈ 20 ✓ compute-optimal")
print("=> 同算力下，Chinchilla 用更小模型+更多数据打败了 GPT-3")

## 4 · 完整成本 drill

把一切串起来：训一个 7B Chinchilla-最优模型，要多少 H100、多久、多少钱。

In [ ]:
N=7e9; D=chinchilla_tokens(N)   # 7B -> 140B token
C=6*N*D
n_gpu=256; gpu="H100"; mfu=0.4
days=C/(HW[gpu]*mfu*n_gpu)/86400
gpu_hours=n_gpu*days*24
cost=gpu_hours*PRICE[gpu]
print(f"训 7B Chinchilla-最优模型 (D={D/1e9:.0f}B token):")
print(f"  FLOPs = {C:.2e}")
print(f"  {n_gpu}×{gpu} @ MFU40%: {days:.1f} 天")
print(f"  GPU-hours = {gpu_hours:,.0f}")
print(f"  成本 ≈ ${cost:,.0f}")
print("\n=> 这套 6ND -> 时间 -> 美元 的链条，是 systems 面试最高频的题")

## 5 · 训练 6ND vs 推理 2N/token（别搞混）

最高频的面试陷阱之一：把训练和推理的 FLOPs 公式混用。推理一次前向 ≈ $2N$ FLOPs/token（只有前向）；训练 ≈ $6ND$（前向 2 + 反向 4，过 $D$ 个 token）。下面在真实模型上把两者算出来，验证训练每 token 恰是推理的 3 倍——用错公式会差出 3 倍。

In [ ]:
# 训练 6ND vs 推理 2N/token：别把两者搞混（高频面试陷阱）。
# 推理一次前向 ≈ 2N FLOPs/token（只有前向，无反向）；训练 ≈ 6ND（前向2+反向4，过 D token）。
def infer_flops_per_token(N): return 2*N
def train_flops_total(N, D): return 6*N*D
N=param_count(load_config("pythia-6.9b"))
inf=infer_flops_per_token(N)
print(f"pythia-6.9b (N={N/1e9:.1f}B):")
print(f"  推理: 每 token ≈ 2N = {inf:.2e} FLOPs")
print(f"  训练: 6ND @ D=300B = {train_flops_total(N,300e9):.2e} FLOPs")
# 训练总量 = 推理单 token 量 × 3D（因为 6ND/(2N)=3D）
ratio = train_flops_total(N,300e9)/inf
print(f"  训练总量 / 推理单token = {ratio:.2e} = 3D = {3*300e9:.2e} ✓")
# 自检：训练每 token(6N) 恰是推理每 token(2N) 的 3 倍
assert abs(6*N/(2*N) - 3) < 1e-9, "训练每 token 是推理的 3 倍(前向+反向)"
assert abs(ratio - 3*300e9) < 1e6
print("\n=> 记牢：训练 6ND、推理 2N/token，差一个反向(2x)的因子；")
print("   面试里把推理成本用 6ND 估会高估 3 倍，反之亦然")

---
## ✏️ 练习区

### ✏️ 练习 1：6ND 训练 FLOPs

实现 `training_flops(N, D)`。验证真实 pythia-6.9b @ 300B token 的量级。

In [ ]:
def training_flops(N, D):
    # TODO: 6*N*D
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
N=param_count(load_config("pythia-6.9b"))
C=training_flops(N, 300e9)
assert 1e22 < C < 2e22, "6.9b @ 300B 应约 1.2e22 FLOPs"
assert abs(training_flops(1e9, 1e9) - 6e18) < 1e12
print(f"练习 1 通过 ✓  pythia-6.9b 训练 = {C:.2e} FLOPs")


### ✏️ 练习 2：训练时间

实现 `training_days(N, D, peak_flops, n_gpu, mfu)`。验证 MFU 越低耗时越长。

In [ ]:
def training_days(N, D, peak_flops, n_gpu, mfu=0.4):
    # TODO: 6ND / (peak*mfu*n_gpu) / 86400
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
N=param_count(load_config("pythia-12b"))
d40=training_days(N, 300e9, HW["A100"], 256, 0.4)
d20=training_days(N, 300e9, HW["A100"], 256, 0.2)
assert abs(d20 - 2*d40) < 1e-6, "MFU 减半，时间翻倍"
assert training_days(N,300e9,HW["H100"],256,0.4) < d40, "H100 更快"
print(f"练习 2 通过 ✓  pythia-12b 256×A100: MFU40%={d40:.1f}天")


### ✏️ 练习 3：Chinchilla 最优与训练充分度

实现 `chinchilla_optimal_tokens(N)`（=20N）和 `undertraining_ratio(N, D_actual)`
（实际 token / 最优 token，<1 表示训练不足）。

In [ ]:
def chinchilla_optimal_tokens(N):
    # TODO: 20*N
    raise NotImplementedError
def undertraining_ratio(N, D_actual):
    # TODO: D_actual / (20*N)
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
assert chinchilla_optimal_tokens(70e9) == 1.4e12
# GPT-3 训练不足
assert undertraining_ratio(175e9, 300e9) < 0.1
# Chinchilla 本身接近最优
assert abs(undertraining_ratio(70e9, 1.4e12) - 1.0) < 0.01
print(f"练习 3 通过 ✓  GPT-3 只训了最优的 {undertraining_ratio(175e9,300e9):.1%}")


### ✏️ 练习 4：完整成本估算

实现 `training_cost_usd(N, D, gpu, n_gpu, mfu, price_per_hr)`：返回美元成本。
（提示：成本 = GPU-hours × 单价；GPU-hours = n_gpu × days × 24）

In [ ]:
def training_cost_usd(N, D, gpu, n_gpu, mfu, price_per_hr):
    # TODO: 用 training_days 算天数 -> GPU-hours -> ×单价
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
# 成本应与卡数无关（更多卡只是更快，总卡时不变）
c1=training_cost_usd(7e9, 140e9, "H100", 128, 0.4, PRICE["H100"])
c2=training_cost_usd(7e9, 140e9, "H100", 512, 0.4, PRICE["H100"])
assert abs(c1-c2)/c1 < 1e-6, "总成本与卡数无关"
# 更大模型更贵
assert training_cost_usd(70e9,1.4e12,"H100",256,0.4,PRICE["H100"]) > c1
print(f"练习 4 通过 ✓  训 7B Chinchilla 模型 ≈ ${c1:,.0f}")


---
## 📖 参考答案

In [ ]:
# 练习 1
def training_flops(N, D): return 6*N*D
print("练习 1 ✓")

In [ ]:
# 练习 2
def training_days(N, D, peak_flops, n_gpu, mfu=0.4):
    return 6*N*D/(peak_flops*mfu*n_gpu)/86400
print("练习 2 ✓")

In [ ]:
# 练习 3
def chinchilla_optimal_tokens(N): return 20*N
def undertraining_ratio(N, D_actual): return D_actual/(20*N)
print("练习 3 ✓")

In [ ]:
# 练习 4
def training_cost_usd(N, D, gpu, n_gpu, mfu, price_per_hr):
    days=training_days(N, D, HW[gpu], n_gpu, mfu)
    return n_gpu*days*24*price_per_hr
print("练习 4 ✓ —— 你现在能口算任何模型的训练账单了。C8 完成！")